# crypto-rmt — demo

Reproduces every figure in the analysis **end to end** from the fetched Binance data. All numerical logic lives in the `crypto_rmt` package; the cells below only load data, call the public API, and save figures into `figures/`.

> Run `python scripts/fetch_data.py` first to populate `data/` (raw files are gitignored).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from crypto_rmt import plotting
from crypto_rmt.clustering import linkage_from_correlation
from crypto_rmt.dynamics import EVENTS, collectivity_series
from crypto_rmt.io import TICKERS, load_prices, returns_matrix
from crypto_rmt.rmt import (
    correlation_matrix,
    eigsystem,
    marchenko_pastur_bounds,
    null_spectrum,
    null_threshold,
    participation_ratio,
)

DATA_DIR = Path('../data')
FIGURES_DIR = Path('../figures')
FIGURES_DIR.mkdir(exist_ok=True)

## Load & align

Timestamp-intersection alignment, then z-scored `log10` returns.

In [ ]:
prices = load_prices(TICKERS, DATA_DIR)
returns, timestamps = returns_matrix(prices, TICKERS, return_timestamps=True)
n_assets, n_obs = returns.shape
print(f'{n_assets} assets x {n_obs} hourly returns')

## Static correlation structure (full window)

Diagonalise the full-window correlation matrix and compare the spectrum to the Marchenko-Pastur noise band.

In [ ]:
C = correlation_matrix(returns)
evals, evecs = eigsystem(C)
lam_minus, lam_plus = marchenko_pastur_bounds(n_assets, n_obs)
print(f'eigenvalue sum = {evals.sum():.2f}  (= N = {n_assets})')
print(f'lambda_max = {evals[0]:.2f}   MP band = [{lam_minus:.3f}, {lam_plus:.3f}]')

> **Note on `T` vs `N`.** With hourly data `T` greatly exceeds `N = 17`, so the Marchenko-Pastur band is very tight and nearly every eigenvalue lands outside it -- strong evidence of genuine structure. The next cell re-estimates the spectrum over a single short window, where the band widens into the textbook MP *bulk*.

In [ ]:
plotting.plot_spectrum_vs_mp(
    evals, n=n_assets, t=n_obs, save=FIGURES_DIR / 'spectrum_vs_mp.png'
)
plt.show()

### The same spectrum over one 30-day window

Estimating `C` over a single 720-hour window (so `T` ~ 720 and the MP band widens to roughly `[0.72, 1.33]`) recovers the textbook picture: a noise **bulk** that most eigenvalues fall inside, with only the market mode (and the odd sector mode) escaping above `lambda_+`. `STATIC_WINDOW` picks the window -- the most recent one here; change the slice for any period.

In [ ]:
STATIC_WINDOW = 720  # one 30-day window; change the slice for a different period
win = returns[:, -STATIC_WINDOW:]
evals_win, _ = eigsystem(correlation_matrix(win))
plotting.plot_spectrum_vs_mp(
    evals_win, n=n_assets, t=STATIC_WINDOW,
    save=FIGURES_DIR / 'spectrum_vs_mp_window.png',
)
plt.show()
lo_w, hi_w = marchenko_pastur_bounds(n_assets, STATIC_WINDOW)
print(
    f'window MP band = [{lo_w:.2f}, {hi_w:.2f}]   '
    f'{int((evals_win > hi_w).sum())} of {n_assets} above lambda+'
)

### Mode participation

How many assets load onto each eigenvector.

In [ ]:
pr = participation_ratio(evecs)
plotting.plot_participation(
    evals, pr, threshold=lam_plus, save=FIGURES_DIR / 'participation.png'
)
plt.show()
print(f'market-mode participation = {pr[0]:.1f} / {n_assets}')

### Correlation cluster map

Hierarchical clustering on the Mantegna distance.

In [ ]:
Z = linkage_from_correlation(C)
plotting.plot_cluster_map(C, Z, TICKERS, save=FIGURES_DIR / 'cluster_map.png')
plt.show()

## Temporal regime detection

Roll a 30-day (720 h) window forward weekly (168 h) and track how collective the market is at each point — the headline of the project.

In [ ]:
end_ts, collectivity, effective_rank = collectivity_series(
    returns, timestamps, window=720, step=168
)
plotting.plot_collectivity(
    end_ts, collectivity, events=EVENTS, save=FIGURES_DIR / 'collectivity.png'
)
plt.show()
peak = int(collectivity.argmax())
print(
    f'collectivity peak = {collectivity[peak]:.2f} on '
    f'{pd.to_datetime(end_ts[peak], unit="s").date()}  '
    f'(median {np.median(collectivity):.2f})'
)

### Collectivity vs effective factor count

The two move inversely across regimes.

In [ ]:
plotting.plot_regime_panel(
    end_ts, collectivity, effective_rank, events=EVENTS,
    save=FIGURES_DIR / 'regime_panel.png',
)
plt.show()

## Diagnostic — why not the shuffle null?

On crypto the correlations are uniformly high, so shuffling the off-diagonals barely lowers the top eigenvalue: the shuffled "null" market mode sits right under the real one and can't separate signal from noise the way Marchenko-Pastur does. Shown here only as a diagnostic; the MP figure above is the one we rely on.

In [ ]:
null_evals = null_spectrum(C, rng=np.random.default_rng(0))
thr = null_threshold(C, rng=np.random.default_rng(0))
plotting.plot_spectrum_vs_null(
    evals, null_evals, thr, save=FIGURES_DIR / 'spectrum_vs_null.png'
)
plt.show()